In [ ]:
# ============================================================
# Part B — Handling Hypothesis Questions
# ============================================================

import numpy as np
import pandas as pd
import joblib
import tensorflow as tf
from sklearn.metrics import accuracy_score

# --------------------------
# 0) Utilities
# --------------------------
def densify(X):
    return X.toarray() if hasattr(X, "toarray") else X #makes it dense

def top2_from_probs(probs_row):
    """
    Returns the indices sorted from smallest to largest
    This being healthy, prediabetes, oral-medicated and insulin
    Then reverse the array
    THen picks the secondlargest probability
    Then returns the index of the two highest probaiblity, what the top is and then the margin
    Finds the most likely class, the second most likely class, and tells you how far apart they are.
    """
    order = np.argsort(probs_row)[::-1]
    top = int(order[0])
    runner = int(order[1])
    return top, runner, float(probs_row[top]), float(probs_row[runner]), float(probs_row[top] - probs_row[runner])


def predict_one(model, preprocessor, row_df):
    if isinstance(row_df, dict):
        row_df = pd.DataFrame([row_df])
    else:
        row_df = row_df.copy()

    # force all column names to strings
    row_df.columns = row_df.columns.map(str)

    # align to exactly what the preprocessor expects
    expected_cols = list(preprocessor.feature_names_in_)
    row_df = row_df.reindex(columns=expected_cols, fill_value=np.nan)

    Xp = densify(preprocessor.transform(row_df))
    probs = model.predict(Xp, verbose=0)[0]
    top, runner, top_conf, runner_conf, margin = top2_from_probs(probs)

    return {
        "top_class": IDX_TO_LABEL[top],
        "top_conf": float(top_conf),
        "runner_class": IDX_TO_LABEL[runner],
        "runner_conf": float(runner_conf),
        "margin": float(margin),
        "probs": {IDX_TO_LABEL[i]: float(probs[i]) for i in range(len(probs))}
    }

In [ ]:
# --------------------------
# 1) Import in the MLP weights (+ preprocessor)
# --------------------------
preprocessor = joblib.load("artifacts/preprocessor.joblib") # preprocessing
class_names  = joblib.load("artifacts/class_names.joblib")  # ["healthy","prediabetes","oral_medicated","insulin_dependent"]

# infer n_in robustly (no .toarray() error)
X1 = preprocessor.transform(X_train.iloc[:1])  # X_train from Part A
n_in = densify(X1).shape[1]

mlp_loaded = build_mlp(n_in=n_in, n_out=4)
mlp_loaded.load_weights("artifacts/final_mlp.weights.h5")
print("Loaded weights. Input dim:", n_in)
# Loads the wights

In [ ]:
# --------------------------
# 2) Apply Features (canonical feature columns = training X columns)
# --------------------------
FEATURE_COLS = X_train.columns.tolist()
print(FEATURE_COLS)

In [ ]:
# --------------------------
# 3) Set Boundaries (uses your BOUNDARIES dict)
# --------------------------
#################################
#Boundaries

BOUNDARIES = {
    "HbA1c": (4, 6.5),
    "Heart Rate Pulse": (60, 100),
    "Oxygen Saturation": (95, 100),
    "Physical Activity": (3867, 10000),
    "Sleep": (0.29, 0.375),
    "Stress": (0, 100),
    "PR Interval": (120, 200),
    "Blood Glucose": (72, 160),
    "Age": (44, 65),
    "RBC": (3.92, 5.65),
    "WBC": (4, 11),
    "BMI": (18.5, 30),
    "Glucose": (62, 125),
    "Height": (161, 175),
    "Cholesterol": (0, 200),
    "Insulin": (0.5, 3),
    "Respiratory Rate": (0, 1),
}
#################################

def boundary_value(col, level="normal"):
    """
    Checks that the column has boundaries
    Extracts the bounds e.g. defines what is low and high
    If level is low will output lo
    If level is high will output hi
    e.g. this translate the question so low HbA1c will output <4
    """
    if col not in BOUNDARIES:
        raise KeyError(f"'{col}' not found in BOUNDARIES")
    lo, hi = BOUNDARIES[col]
    if level == "low":
        return float(lo)
    if level == "high":
        return float(hi)
    return float((lo + hi) / 2)

    # numeric vars = boundary keys present in master

numeric_query_vars = [k for k in BOUNDARIES.keys() if k in master.columns and k not in drop_cols]

condition_vars = [
    c for c in master.columns
    if c not in drop_cols
    and master[c].dropna().isin([0, 1]).all()
]

condition_vars = list(set(condition_vars))

In [ ]:
# --------------------------
# 4.1) Create Standard Patient (midpoint of each boundary everything else 0)
# --------------------------

def is_binary_column(df, col):
    """
    Checks whether the column is a binary feature e.g. the categories
    Removes missing Values
    Makes sure Column isn't empty
    """
    vals = set(df[col].dropna().unique())
    return vals.issubset({0, 1}) and len(vals) > 0


def make_standard_patient(df_raw, feature_cols):
    """
    Baseline logic:
      - binary column (0/1) -> 0
      - numeric + boundary -> midpoint
      - numeric no boundary -> dataset median
      - otherwise -> 0
    """
    row = {}

    for col in feature_cols: #looks through the model's training column

        # If column doesn't exist in raw df
        if col not in df_raw.columns:
            row[col] = 0.0
            continue

        if is_binary_column(df_raw, col): #makes sure won't have any other categories
            row[col] = 0.0

        elif col in BOUNDARIES:
            lo, hi = BOUNDARIES[col]
            row[col] = float((lo + hi) / 2)

        elif np.issubdtype(df_raw[col].dtype, np.number):
            row[col] = float(df_raw[col].median())

        else:
            row[col] = 0.0

    return pd.DataFrame([row], columns=feature_cols)

standard_patient = make_standard_patient(df, FEATURE_COLS)

print("Standard patient created.")
print(standard_patient)

In [ ]:
# --------------------------
# 4.2) Create Standard Patients (for each subgroup)
# --------------------------

def get_true_label_index(df_raw):
    # Converts the 4 on-hot columns into a single integer class label 0,1,2,3
    return df_raw[target_cols].astype(int).values.argmax(axis=1)


def make_standard_subgroup(df_sub, feature_cols):
    """
    Makes a standard patient for each class -> healthy, prediabetes, oral_medicated, insulin
    mode for binary, median for numeric.
    """
    row = {}
    for c in feature_cols:
        if c not in df_sub.columns:
            row[c] = 0.0
            continue

        s = df_sub[c]
        if is_binary_column(df_sub, c):
            # mode for binary e.g. what comes up the most
            vals = s.dropna()
            row[c] = float(vals.mode().iloc[0]) if len(vals) else 0.0
        else:
            # median for numeric e.g. adding up all the values so to find an average
            row[c] = float(pd.to_numeric(s, errors="coerce").median())
    return pd.DataFrame([row], columns=feature_cols)

def build_prototypes_by_true_group(df_raw):
    """
    Splits the dataset into true label groups
    Build prototype one per group
    So will pick only the patients fomr the TRUE class
    Creates a dictionary with healthy, prediabetes, oral_medicated, insulin_dependent

    """
    y_idx = get_true_label_index(df_raw)
    prototypes = {}

    for k, name in enumerate(class_names):
        grp = df_raw.loc[y_idx == k].copy()
        prototypes[name] = make_standard_subgroup(grp, FEATURE_COLS)

    return prototypes

prototypes = build_prototypes_by_true_group(df)
print("Standard Subgroups:", list(prototypes.keys()))

In [ ]:
# --------------------------
# 4.3) Predict fake patient with MLP
# --------------------------
def predict_one(model, preprocessor, row_df):
    Xp = densify(preprocessor.transform(row_df))
    probs = model.predict(Xp, verbose=0)[0]
    top, runner, top_conf, runner_conf, margin = top2_from_probs(probs)
    return {
        "probs": probs,
        "top_idx": top,
        "top_class": class_names[top],
        "top_conf": top_conf,
        "runner_class": class_names[runner],
        "runner_conf": runner_conf,
        "margin": margin,
    }

In [ ]:
# --------------------------
# 5) Ask Question
# Mode: mixed
# Question: Does HIGH Physical Activity and Type Ii Diabetes affect likelihood of T2DM category?
# --------------------------
def manual_query(master, numeric_vars, condition_vars, mode="manual"):
    """
    Manual version of random_query.
    Returns: (var1, level1, var2, level2, mode_used)
    """

    available_modes = ["numeric2", "cond2", "mixed", "single"]

    print("Available modes:")
    for m in available_modes:
        print("-", m)

    while True:
        mode_used = input("Choose mode: ").strip().lower()
        if mode_used in available_modes:
            break
        print(f"❌ Invalid mode. Choose one of: {available_modes}")

    if mode_used == "single":
        while True:
            kind = input("Choose feature type ('numeric' or 'condition'): ").strip().lower()
            if kind in ["numeric", "condition"]:
                break
            print("❌ Invalid choice. Type 'numeric' or 'condition'.")

        if kind == "numeric":
            print("\nNumeric features:")
            for f in numeric_vars:
                print("-", f)

            while True:
                var1 = input("Choose numeric feature: ").strip()
                if var1 in numeric_vars:
                    break
                print("❌ Invalid feature. Please choose from the list.")

            valid_levels = ["LOW", "NORMAL", "HIGH"]
            while True:
                level1 = input("Choose level (LOW / NORMAL / HIGH): ").strip().upper()
                if level1 in valid_levels:
                    break
                print("❌ Invalid level. Choose LOW, NORMAL, or HIGH.")

            return var1, level1, None, None, "single"

        elif kind == "condition":
            print("\nCondition features:")
            for f in condition_vars:
                print("-", f)

            while True:
                var1 = input("Choose condition feature: ").strip()
                if var1 in condition_vars:
                    break
                print("❌ Invalid condition. Please choose from the list.")

            return var1, None, None, None, "single"

    if mode_used == "numeric2":
        print("\nNumeric features:")
        for f in numeric_vars:
            print("-", f)

        while True:
            var1 = input("Choose first numeric feature: ").strip()
            if var1 in numeric_vars:
                break
            print("❌ Invalid feature. Please choose from the list.")

        while True:
            level1 = input("Choose first level (LOW / HIGH): ").strip().upper()
            if level1 in ["LOW", "HIGH"]:
                break
            print("❌ Invalid level. Choose LOW or HIGH.")

        while True:
            var2 = input("Choose second numeric feature: ").strip()
            if var2 in numeric_vars:
                break
            print("❌ Invalid feature. Please choose from the list.")

        while True:
            level2 = input("Choose second level (LOW / HIGH): ").strip().upper()
            if level2 in ["LOW", "HIGH"]:
                break
            print("❌ Invalid level. Choose LOW or HIGH.")

        return var1, level1, var2, level2, "numeric2"

    if mode_used == "cond2":
        print("\nCondition features:")
        for f in condition_vars:
            print("-", f)

        while True:
            var1 = input("Choose first condition feature: ").strip()
            if var1 in condition_vars:
                break
            print("❌ Invalid condition. Please choose from the list.")

        while True:
            var2 = input("Choose second condition feature: ").strip()
            if var2 in condition_vars:
                break
            print("❌ Invalid condition. Please choose from the list.")

        return var1, None, var2, None, "cond2"

    if mode_used == "mixed":
        print("\nNumeric features:")
        for f in numeric_vars:
            print("-", f)

        while True:
            var1 = input("Choose numeric feature: ").strip()
            if var1 in numeric_vars:
                break
            print("❌ Invalid feature. Try again.")

        while True:
            level1 = input("Choose level (LOW / HIGH): ").strip().upper()
            if level1 in ["LOW", "HIGH"]:
                break
            print("❌ Invalid level. Choose LOW or HIGH.")

        print("\nCondition features:")
        for f in condition_vars:
            print("-", f)

        while True:
            var2 = input("Choose condition feature: ").strip()
            if var2 in condition_vars:
                break
            print("❌ Invalid condition. Try again.")

        return var1, level1, var2, None, "mixed"

# Ask Question

In [ ]:
# Generate a hypothesis
var1, level1, var2, level2, mode_used = manual_query(
    master=master,
    numeric_vars=numeric_query_vars,
    condition_vars=condition_vars
)

In [ ]:
# --------------------------
# 6) Change Standard Patient columns for the hypothesis
#    HIGH Physical Activity -> set to boundary high
#    Type Ii Diabetes -> set to 1 (flag)
# --------------------------
def changes_for_var(var, level, condition_vars):
    """
    Returns the 'changes' list used by apply_changes.
    - If level is not None -> numeric boundary variable (LOW/NORMAL/HIGH)
    - Else -> binary variable, set to 1
    """
    if level is None:
        return [{"col": var, "type": "value", "value": 1.0}]
    else:
        return [{"col": var, "type": "boundary", "level": str(level).lower()}]  # expects "low/high/normal"




In [ ]:
# Convert variables into change instructions
changes1 = changes_for_var(var1, level1, condition_vars)

changes2 = []
if var2 is not None:
    changes2 = changes_for_var(var2, level2, condition_vars)

hypothesis_changes = changes1 + changes2

print("Hypothesis changes:", hypothesis_changes)

In [ ]:
# --------------------------
# 7.1) Sanity Checking Standard Patients (for each subgroup)
# --------------------------
def predict_label_and_probs(row_df):
    res = predict_one(mlp_loaded, preprocessor, row_df)
    return res["top_class"], res["top_conf"], res["probs"]

print("\nPrototype sanity check:")
for name, proto in prototypes.items():
    pred, conf, _ = predict_label_and_probs(proto)
    print(f"Prototype built from TRUE {name:>16} -> model predicts {pred:>16} (conf={conf:.2f})")

In [ ]:
def apply_changes(patient_df, changes):
    out = patient_df.copy()
    for ch in changes:
        col = ch["col"]
        if ch.get("type") == "boundary":
            out.loc[:, col] = boundary_value(col, ch["level"])
        else:
            out.loc[:, col] = ch["value"]
    return out

In [ ]:
# --------------------------
# 7.2) Run hypothesis on all prototypes
# --------------------------
def run_hypothesis_on_all_prototypes(prototypes, changes):
    results = []

    for group_name, proto in prototypes.items():

        base_res = predict_one(mlp_loaded, preprocessor, proto)

        changed = apply_changes(proto, changes)
        changed_res = predict_one(mlp_loaded, preprocessor, changed)

        delta = changed_res["probs"] - base_res["probs"]

        results.append({
            "subgroup": group_name,   # ← renamed here
            "baseline_pred": base_res["top_class"],
            "baseline_conf": base_res["top_conf"],
            "changed_pred": changed_res["top_class"],
            "changed_conf": changed_res["top_conf"],
            "delta_healthy": float(delta[0]),
            "delta_prediabetes": float(delta[1]),
            "delta_oral_medicated": float(delta[2]),
            "delta_insulin_dependent": float(delta[3]),
        })

    return pd.DataFrame(results)

res_df = run_hypothesis_on_all_prototypes(prototypes, hypothesis_changes)
res_df

In [ ]:
# --------------------------
# 8) Find all dataset patients that fit the hypothesis
#    HIGH Physical Activity = >= upper boundary
#    Type Ii Diabetes = 1
# --------------------------
def subset_dataset(df_raw, changes):
    mask = pd.Series(True, index=df_raw.index)
    for ch in changes:
        col = ch["col"]
        if col not in df_raw.columns:
            raise KeyError(f"Column '{col}' not in df.")
        if ch.get("type") == "boundary":
            lo, hi = BOUNDARIES[col]
            if ch["level"] == "high":
                mask &= df_raw[col].astype(float) >= float(hi)
            elif ch["level"] == "low":
                mask &= df_raw[col].astype(float) <= float(lo)
            else:
                mask &= df_raw[col].astype(float).between(float(lo), float(hi), inclusive="both")
        else:
            mask &= (df_raw[col].astype(float) == float(ch["value"]))
    return df_raw[mask].copy()

sub_df = subset_dataset(df, hypothesis_changes)   # df is your full raw dataset
n_sub = len(sub_df)

# predict subgroup (if any)
if n_sub > 0:
    X_sub = sub_df[FEATURE_COLS].copy()
    X_sub_p = densify(preprocessor.transform(X_sub))
    sub_probs = mlp_loaded.predict(X_sub_p, verbose=0)
    sub_pred = sub_probs.argmax(axis=1)

    # distribution + majority
    counts = np.bincount(sub_pred, minlength=4)
    majority_idx = int(np.argmax(counts))
    majority_name = class_names[majority_idx]
    majority_share = float(counts[majority_idx] / n_sub)

    # confidence summaries
    mean_top_conf = float(np.max(sub_probs, axis=1).mean())
    order = np.argsort(sub_probs, axis=1)[:, ::-1]
    margins = sub_probs[np.arange(n_sub), order[:, 0]] - sub_probs[np.arange(n_sub), order[:, 1]]
    mean_margin = float(margins.mean())
else:
    majority_name = None
    majority_share = None
    mean_top_conf = None
    mean_margin = None

In [ ]:
prototypes = build_prototypes_by_true_group(df)  

In [ ]:
def describe_var(var, level, condition_vars):
    """
    Turn a variable + level into readable text for the question header.
    """
    if level is None:
        return str(var)
    return f"{str(level).upper()} {str(var)}"

In [ ]:
# --------------------------
# 9) Output Sentence Block (baseline + fake + dataset)
# --------------------------


def class_distribution_text(pred_labels, class_names):
    counts = np.bincount(pred_labels, minlength=len(class_names))
    total = counts.sum()
    order = np.argsort(counts)[::-1]

    lines = []
    for idx in order:
        if counts[idx] == 0:
            continue
        pct = counts[idx] / total
        lines.append(f"{class_names[idx]}: {counts[idx]}/{total} ({pct:.1%})")

    return ", ".join(lines)


def predicted_class_condition_breakdown(all_pred, sub_pred, class_names):
    """
    For each predicted class in the full dataset:
    how many of those predicted patients fall inside the condition
    """
    total_counts = np.bincount(all_pred, minlength=len(class_names))
    sub_counts = np.bincount(sub_pred, minlength=len(class_names))

    lines = []
    for i, name in enumerate(class_names):
        total = total_counts[i]
        in_condition = sub_counts[i]
        pct = 0.0 if total == 0 else in_condition / total
        lines.append(f"{name}: {in_condition}/{total} ({pct:.1%})")
    return lines
    
def compare_independent_effects(
    df,
    var1, level1,      # example: "Physical Activity", "HIGH"
    var2, level2,      # example: "Type Ii Diabetes", None
    condition_vars,    # set of binary 0/1 columns
    preprocessor,
    mlp,
    drop_cols,
    test_acc,
    prototypes=None,
    show_joint=True
):
    X_all = df[FEATURE_COLS].copy()
    X_all_p = densify(preprocessor.transform(X_all))
    all_pred = mlp.predict(X_all_p, verbose=0).argmax(axis=1)

    # 1) Base Patient
    # -------------------------
    # QUESTION HEADER (NEW)
    # -------------------------

    X_all = df[FEATURE_COLS].copy()

    s1 = describe_var(var1, level1, set(condition_vars))

    if var2 is None:
        question_line = f"Does {s1} affect likelihood of T2DM category?"
    else:
        s2 = describe_var(var2, level2, set(condition_vars))
        question_line = f"Does {s1} AND {s2} affect likelihood of T2DM category?"
              
    
    base_patient = make_standard_patient(df, FEATURE_COLS)
    base_res = predict_one(mlp, preprocessor, base_patient)

    # 1.1) Fake Patient(Median)
    # Base fake patient (median/midpoint baseline)
    base_patient = make_standard_patient(df, FEATURE_COLS)

    # Build changes for question

    changes1 = changes_for_var(var1, level1, condition_vars)

    changes2 = []
    if var2 is not None:
        changes2 = changes_for_var(var2, level2, condition_vars)
    
    changes_joint = changes1 + changes2


    # Fake patient that matches the full question (joint)
    fake_question_patient = apply_changes(base_patient, changes_joint)
    fake_question_res = predict_one(mlp, preprocessor, fake_question_patient)

    fake_aligned = fake_question_patient.reindex(columns=base_patient.columns, fill_value=np.nan)

    changed_cols = fake_aligned.iloc[0].ne(base_patient.iloc[0])
    print("Changed columns:", list(base_patient.columns[changed_cols]))

    # 2) Changed Patient (Var1 only)
    changes1 = []
    if level1 is not None:  # var1 is numeric
        changes1.append({"col": var1, "type": "boundary", "level": level1})
    else: # var1 is binary
        changes1.append({"col": var1, "type": "value", "value": 1.0})

    patient1 = apply_changes(base_patient, changes1)
    res1 = predict_one(mlp, preprocessor, patient1)

    
    # 3) Changed Patient (Var2 only)

    res2 = None
    sub_df2 = pd.DataFrame()
    
    if var2 is not None:
        changes2 = []
        if level2 is not None:
            changes2.append({"col": var2, "type": "boundary", "level": level2})
        else:
            changes2.append({"col": var2, "type": "value", "value": 1.0})
    
        patient2 = apply_changes(base_patient, changes2)
        res2 = predict_one(mlp, preprocessor, patient2)
    
    # 4) Changed Patient (Joint)
    if show_joint:
        changes_joint = changes1 + changes2
        patient_joint = apply_changes(base_patient, changes_joint)
        res_joint = predict_one(mlp, preprocessor, patient_joint)

    # 5) Subsetting the dataset
    sub_df1 = subset_dataset(df, changes1)
    sub_df2 = subset_dataset(df, changes2)
    sub_df_joint = subset_dataset(df, changes_joint) if show_joint else pd.DataFrame()

    def get_impact_description(delta_probs):
        if abs(delta_probs) > 0.3: return "High"
        if abs(delta_probs) > 0.1: return "Moderate"
        return "Low"

    def get_confidence_description(conf):
        if conf > 0.8: return "High"
        if conf > 0.5: return "Moderate"
        return "Low"

    report = question_line
    report += f"Baseline prediction for average patient: {base_res['top_class']} (conf={base_res['top_conf']:.2f})\n\n"

    # Var1 report
    var1_desc = f"{level1 or ''} {var1}".strip()
    report += f"Effect of {var1_desc}:\n"
    report += f"  Predicted class: {res1['top_class']} (conf={res1['top_conf']:.2f})\n"
    report += f"  Delta to baseline: {get_impact_description(res1['top_conf'] - base_res['top_conf'])} ({res1['top_conf'] - base_res['top_conf']:.2f})\n"
    
    if not sub_df1.empty:
        X_sub1_p = densify(preprocessor.transform(sub_df1[FEATURE_COLS]))
        sub_pred1 = mlp.predict(X_sub1_p, verbose=0).argmax(axis=1)
    
        dist_text = class_distribution_text(sub_pred1, class_names)
        report += (
            f"  Among patients with {var1_desc} ({len(sub_df1)} patients), "
            f"the predicted class distribution is: {dist_text}.\n"
        )
    
       # lines1 = predicted_class_condition_breakdown(all_pred, sub_pred1, class_names)
        #report += f"  Out of all predicted patients:\n"
        #for line in lines1:
         #   report += f"    - {line} have {var1_desc}\n"
        #report += "\n"
    #else:
     #   report += f"  No patients found in dataset with {var1_desc}.\n\n"
    

    # Var2 report
    if var2 is not None:
        var2_desc = f"{level2 or ''} {var2}".strip()
        report += f"Effect of {var2_desc}:\n"
        report += f"  Predicted class: {res2['top_class']} (conf={res2['top_conf']:.2f})\n"
        report += f"  Delta to baseline: {get_impact_description(res2['top_conf'] - base_res['top_conf'])} ({res2['top_conf'] - base_res['top_conf']:.2f})\n"
    
        if not sub_df2.empty:
            X_sub2_p = densify(preprocessor.transform(sub_df2[FEATURE_COLS]))
            sub_pred2 = mlp.predict(X_sub2_p, verbose=0).argmax(axis=1)
    
            dist_text = class_distribution_text(sub_pred2, class_names)
            report += (
                f"  Among patients with {var2_desc} ({len(sub_df2)} patients), "
                f"the predicted class distribution is: {dist_text}.\n"
            )
    
            lines2 = predicted_class_condition_breakdown(all_pred, sub_pred2, class_names)
    # Joint report
    if show_joint and var2 is not None:
        joint_desc = f"{var1_desc} AND {var2_desc}"
        report += f"Combined effect of {joint_desc}:\n"
        report += f"  Predicted class: {res_joint['top_class']} (conf={res_joint['top_conf']:.2f})\n"
        report += f"  Delta to baseline: {get_impact_description(res_joint['top_conf'] - base_res['top_conf'])} ({res_joint['top_conf'] - base_res['top_conf']:.2f})\n"
    
        if not sub_df_joint.empty:
            X_sub_joint_p = densify(preprocessor.transform(sub_df_joint[FEATURE_COLS]))
            sub_pred_joint = mlp.predict(X_sub_joint_p, verbose=0).argmax(axis=1)
    
            dist_text = class_distribution_text(sub_pred_joint, class_names)
            report += (
                f"  Among patients with {joint_desc} ({len(sub_df_joint)} patients), "
                f"the predicted class distribution is: {dist_text}.\n"
            )
    
            lines_joint = predicted_class_condition_breakdown(all_pred, sub_pred_joint, class_names)
           # report += f"  Out of all predicted patients:\n"
           # for line in lines_joint:
            #    report += f"    - {line} have {joint_desc}\n"
            #report += "\n"
        #else:
         #   report += f"  No patients found in dataset with {joint_desc}.\n\n"

    
    # -------------------------
    # Subgroup prototype fake patients (NEW)
    # -------------------------
    if prototypes is not None:
        report += "Fake patients by TRUE subgroup (before → after applying question):\n"

        for true_group_name, proto in prototypes.items():
            base_p = predict_one(mlp, preprocessor, proto)

            changed_proto = apply_changes(proto, changes_joint)
            changed_p = predict_one(mlp, preprocessor, changed_proto)

            report += (
                f"  - Prototype TRUE {true_group_name}: "
                f"{base_p['top_class']} ({base_p['top_conf']:.2f}) → "
                f"{changed_p['top_class']} ({changed_p['top_conf']:.2f}) "
                f"[Δhealthy={changed_p['probs'][0]-base_p['probs'][0]:+.2f}, "
                f"Δpred={changed_p['probs'][1]-base_p['probs'][1]:+.2f}, "
                f"Δoral={changed_p['probs'][2]-base_p['probs'][2]:+.2f}, "
                f"Δins={changed_p['probs'][3]-base_p['probs'][3]:+.2f}]"
                "\n"
            )

        report += "\n"

    
    return report, base_res, fake_question_res, sub_df_joint, 5

import random
import pandas as pd
import json

def run_n_manual_questions(
    master,
    numeric_vars,
    condition_vars,
    preprocessor, mlp, drop_cols,
    test_acc,
    n=5,
    show_joint=True,
):
    results = []

    for i in range(1, n + 1):
        print("\n" + "=" * 70)
        print(f"Question {i}")
        print("=" * 70)

        var1, level1, var2, level2, mode_used = manual_query(
            master=master,
            numeric_vars=numeric_vars,
            condition_vars=condition_vars
        )

        s1 = describe_var(var1, level1, set(condition_vars))

        if var2 is None:
            question_text = f"Does {s1} affect likelihood of T2DM category?"
        else:
            s2 = describe_var(var2, level2, set(condition_vars))
            question_text = f"Does {s1} AND {s2} affect likelihood of T2DM category?"

        report, base_res, fake_question_res, sub_df_joint, extra_val = compare_independent_effects(
            df=master,
            var1=var1, level1=level1,
            var2=var2, level2=level2,
            condition_vars=set(condition_vars),
            preprocessor=preprocessor,
            mlp=mlp,
            drop_cols=drop_cols,
            test_acc=test_acc,
            show_joint=show_joint,
            prototypes=prototypes
        )

        results.append({
            "run_id": i,
            "mode_used": mode_used,
            "var1": var1,
            "level1": level1,
            "var2": var2,
            "level2": level2,
            "question_text": question_text,
            "report": report,
            "baseline_top_class": base_res["top_class"],
            "baseline_top_conf": base_res["top_conf"],
            "fake_top_class": fake_question_res["top_class"],
            "fake_top_conf": fake_question_res["top_conf"],
            "runner_up_class": fake_question_res["runner_class"],
            "margin": fake_question_res["margin"],
            "subset_joint_size": len(sub_df_joint) if sub_df_joint is not None else 0
        })

    return results
    
results = run_n_manual_questions(
    master=master,
    numeric_vars=numeric_query_vars,
    condition_vars=condition_vars,
    preprocessor=preprocessor,
    mlp=mlp_loaded,
    drop_cols=drop_cols,
    test_acc=test_acc,
    n=1,
    show_joint=True
)

for r in results:
    print("\n" + "=" * 60)
    print(f"Question {r['run_id']} | Mode={r['mode_used']}")
    print("Question:", r["question_text"])
    print()
    print(r["report"])

df_results = pd.DataFrame(results)
df_results.to_csv("random_question_results.csv", index=False)

In [ ]:
report, base_res, fake_question_res, sub_df_joint, proto_changes = compare_independent_effects(
    df=master,
    var1=r["var1"], level1=r["level1"],
    var2=r["var2"], level2=r["level2"],
    condition_vars=set(condition_vars),
    preprocessor=preprocessor,
    mlp=mlp_loaded,
    drop_cols=drop_cols,
    test_acc=test_acc,
    show_joint=True
)

r = results[0]

results.append({
    "i": i,
    "mode_used": mode_used,
    "var1": var1,
    "level1": level1,
    "var2": var2,
    "level2": level2,
    "report": report
})

In [ ]:
import numpy as np

# ===========================
# Conclusion template builder
# ===========================

def severity_from_probs(probs_row):
    """
    Severity = P(prediabetes) + P(oral_medicated) + P(insulin_dependent)
    Assumes class order: [healthy, prediabetes, oral_medicated, insulin_dependent]
    """
    p = np.asarray(probs_row, dtype=float)
    return float(p[1] + p[2] + p[3])


def describer1_from_delta_sev(delta_sev):
    # Effect direction/strength on severity
    if delta_sev < -0.15: return "very strongly decreases"
    if delta_sev < -0.05: return "decreases"
    if delta_sev <= 0.05: return "is unlikely to change"
    if delta_sev <= 0.15: return "increases"
    return "very strongly increases"


def describer2_from_max_shift(max_shift):
    # Probability shift magnitude
    if max_shift < 0.03: return "minimal"
    if max_shift < 0.10: return "mixed"
    return "substantial"


def describer4_from_transitions(n_transitions):
    # Class transitions count
    if n_transitions == 0: return "no"
    if n_transitions <= 2: return "some"
    return "many"


def describer5_from_delta_sev(delta_sev):
    # Increase in severity magnitude (non-negative framing)
    if delta_sev <= 0.0: return "no"
    if delta_sev < 0.05: return "minimal"
    if delta_sev < 0.15: return "moderate"
    return "large"


def describer6_from_n(n):
    # Evidential strength from subgroup size
    if n is None: return "unknown"
    if n < 20: return "weak"
    if n < 100: return "moderate"
    return "strong"


def describer7_final_likelihood(delta_sev, n_transitions, n_joint, tol=0.05):
    """
    Final likelihood of meaningful effect:
      - downweight if subgroup is tiny
      - upweight if severity delta is large and many transitions
    """
    if n_joint is None:
        return "unknown"

    # tiny subgroup => cap
    tiny = (n_joint < 20)

    # baseline by severity
    if abs(delta_sev) <= tol and n_transitions == 0:
        base = "unlikely"
    elif abs(delta_sev) <= 0.15 and n_transitions <= 2:
        base = "likely"
    else:
        base = "very likely"

    if tiny and base == "very likely":
        return "likely"  # cap due to low evidence
    if tiny and base == "likely":
        return "unlikely"  # cap further

    return base


def prediction_certainty(top_conf, margin, conf_thr=0.75, margin_thr=0.20):
    if top_conf >= conf_thr and margin >= margin_thr:
        return "High"
    if top_conf >= 0.55 and margin >= 0.10:
        return "Moderate"
    return "Low"


def model_reliability_from_acc(test_acc):
    if test_acc is None:
        return "Unknown"
    if test_acc < 0.60:
        return "Low"
    if test_acc < 0.75:
        return "Moderate"
    return "High"


def build_conclusion_block(
    question_variables,          # str e.g. "HbA1c=LOW AND Oxygen Saturation=LOW"
    base_probs,                  # np.array shape (4,)
    changed_probs,               # np.array shape (4,)
    changed_top_class,           # bool
    changed_top_conf,            # float
    changed_margin,              # float
    joint_subgroup_n=None,       # int
    prototype_transition_count=0,# int 0..4
    test_acc=None                # float
):
    """
    Returns a formatted conclusion block matching your template.
    """
    base_probs = np.asarray(base_probs, dtype=float)
    changed_probs = np.asarray(changed_probs, dtype=float)

    sev_base = severity_from_probs(base_probs)
    sev_changed = severity_from_probs(changed_probs)
    delta_sev = sev_changed - sev_base

    max_shift = float(np.max(np.abs(changed_probs - base_probs)))

    # transitions: fake patient class change counts as 1, plus prototype transitions
    n_transitions = int(changed_top_class) + int(prototype_transition_count)

    d1 = describer1_from_delta_sev(delta_sev)
    d2 = describer2_from_max_shift(max_shift)
    d4 = describer4_from_transitions(n_transitions)
    d5 = describer5_from_delta_sev(delta_sev)
    d6 = describer6_from_n(joint_subgroup_n)
    d7 = describer7_final_likelihood(delta_sev, n_transitions, joint_subgroup_n)

    pred_cert = prediction_certainty(changed_top_conf, changed_margin)
    model_rel = model_reliability_from_acc(test_acc)

    # Build text
    lines = []
    lines.append("=== Conclusion ===============================\n")
    lines.append("In this model,")
    lines.append(f"({question_variables}) {d1} the predicted T2DM severity")
    lines.append(f"(Severity: {sev_base:.3f} → {sev_changed:.3f}, Δ={delta_sev:+.3f}).\n")
    lines.append("Both controlled model probes and real-patient subgroup analysis showed")
    lines.append(f"({d2}) probability shifts, ({d4}) class transitions, and ({d5}) increase in severity.")
    lines.append(f"The subgroup size (n={joint_subgroup_n}) provides ({d6}) evidential strength.")
    lines.append(f"Therefore, this hypothesis is **{d7.upper()}** to meaningfully affect likelihood of T2DM.\n")
    lines.append(f"Prediction certainty: {pred_cert} (top_conf={changed_top_conf:.2f}, margin={changed_margin:.2f})")
    lines.append(f"Model reliability: {model_rel}" + (f" (test_acc={test_acc:.2f})" if test_acc is not None else ""))
    lines.append("\n==============================================")

    return "\n".join(lines)

In [ ]:
import numpy as np

# ===========================
# Conclusion template builder
# ===========================

def severity_from_probs(probs_row):
    """
    Severity = P(prediabetes) + P(oral_medicated) + P(insulin_dependent)
    Assumes class order: [healthy, prediabetes, oral_medicated, insulin_dependent]
    """
    p = np.asarray(probs_row, dtype=float)
    return float(p[1] + p[2] + p[3])


def describer1_from_delta_sev(delta_sev):
    if delta_sev < -0.15: return "very strongly decreases"
    if delta_sev < -0.05: return "decreases"
    if delta_sev <= 0.05: return "is unlikely to change"
    if delta_sev <= 0.15: return "increases"
    return "very strongly increases"


def describer2_from_max_shift(max_shift):
    if max_shift < 0.03: return "minimal"
    if max_shift < 0.10: return "mixed"
    return "substantial"


def describer4_from_transitions(n_transitions):
    if n_transitions == 0: return "no"
    if n_transitions <= 2: return "some"
    return "many"


def describer5_from_delta_sev(delta_sev):
    if delta_sev <= 0.0: return "no"
    if delta_sev < 0.05: return "minimal"
    if delta_sev < 0.15: return "moderate"
    return "large"


def describer6_from_n(n):
    if n is None: return "unknown"
    if n < 20: return "weak"
    if n < 100: return "moderate"
    return "strong"


def describer7_final_likelihood(delta_sev, n_transitions, n_joint, tol=0.05):
    if n_joint is None:
        return "unknown"

    tiny = (n_joint < 20)

    if abs(delta_sev) <= tol and n_transitions == 0:
        base = "unlikely"
    elif abs(delta_sev) <= 0.15 and n_transitions <= 2:
        base = "likely"
    else:
        base = "very likely"

    if tiny and base == "very likely":
        return "likely"
    if tiny and base == "likely":
        return "unlikely"

    return base


def prediction_certainty(top_conf, margin, conf_thr=0.75, margin_thr=0.20):
    if top_conf >= conf_thr and margin >= margin_thr:
        return "High"
    if top_conf >= 0.55 and margin >= 0.10:
        return "Moderate"
    return "Low"


def model_reliability_from_acc(test_acc):
    if test_acc is None:
        return "Unknown"
    if test_acc < 0.60:
        return "Low"
    if test_acc < 0.75:
        return "Moderate"
    return "High"


def build_conclusion_block(
    question_variables,
    base_probs,
    changed_probs,
    base_top_class,              # str, e.g. "healthy"
    changed_top_class,           # str, e.g. "prediabetes"
    changed_top_conf,
    changed_margin,
    joint_subgroup_n=None,
    prototype_transition_count=0,
    test_acc=None
):
    """
    Returns a formatted conclusion block matching your template.
    """
    base_probs = np.asarray(base_probs, dtype=float)
    changed_probs = np.asarray(changed_probs, dtype=float)

    sev_base = severity_from_probs(base_probs)
    sev_changed = severity_from_probs(changed_probs)
    delta_sev = sev_changed - sev_base

    max_shift = float(np.max(np.abs(changed_probs - base_probs)))

    # 1 if predicted class changed, else 0
    class_changed = int(base_top_class != changed_top_class)
    n_transitions = class_changed + int(prototype_transition_count)

    d1 = describer1_from_delta_sev(delta_sev)
    d2 = describer2_from_max_shift(max_shift)
    d4 = describer4_from_transitions(n_transitions)
    d5 = describer5_from_delta_sev(delta_sev)
    d6 = describer6_from_n(joint_subgroup_n)
    d7 = describer7_final_likelihood(delta_sev, n_transitions, joint_subgroup_n)

    pred_cert = prediction_certainty(changed_top_conf, changed_margin)
    model_rel = model_reliability_from_acc(test_acc)

    lines = []
    lines.append("=== Conclusion ===============================\n")
    lines.append("In this model,")
    lines.append(f"({question_variables}) {d1} the predicted T2DM severity")
    lines.append(f"(Severity: {sev_base:.3f} → {sev_changed:.3f}, Δ={delta_sev:+.3f}).\n")
    lines.append("Both controlled model probes and real-patient subgroup analysis showed")
    lines.append(f"({d2}) probability shifts, ({d4}) class transitions, and ({d5}) increase in severity.")
    lines.append(f"The subgroup size (n={joint_subgroup_n}) provides ({d6}) evidential strength.")
    lines.append(f"Therefore, this hypothesis is **{d7.upper()}** to meaningfully affect likelihood of T2DM.\n")
    lines.append(f"Prediction certainty: {pred_cert} (top_conf={changed_top_conf:.2f}, margin={changed_margin:.2f})")
    lines.append(f"Model reliability: {model_rel}" + (f" (test_acc={test_acc:.2f})" if test_acc is not None else ""))
    lines.append("\n==============================================")

    return "\n".join(lines)

In [ ]:
PartA_conclusion_results = []

r = results[0]
idx = 1

report, base_res, fake_question_res, sub_df_joint, proto_changes = compare_independent_effects(
        df=master,
        var1=r["var1"], level1=r["level1"],
        var2=r["var2"], level2=r["level2"],
        condition_vars=set(condition_vars),
        preprocessor=preprocessor,
        mlp=mlp_loaded,
        drop_cols=drop_cols,
        test_acc=test_acc,
        show_joint=True
    )

question_vars = f"{r['var1']}={r['level1']} AND {r['var2']}={r['level2']}"

conclusion_text = build_conclusion_block(
        question_variables=question_vars,
        base_probs=base_res["probs"],
        changed_probs=fake_question_res["probs"],
        base_top_class=base_res["top_class"],
        changed_top_class=fake_question_res["top_class"],
        changed_top_conf=fake_question_res["top_conf"],
        changed_margin=fake_question_res["margin"],
        joint_subgroup_n=len(sub_df_joint),
        prototype_transition_count=proto_changes,
        test_acc=test_acc
    )
 
print("\n" + "="*60)
print(f"Question {idx}")
print(question_vars)
print("="*60)
print(conclusion_text)

PartA_conclusion_results.append({
        "question_num": idx,
        "question_vars": question_vars,
        "conclusion": conclusion_text
    })